# Experiment 001, Typo Robustness: Colab driver

Four steps: **bootstrap → configure → run → download**. Every pipeline stage
(data prep, generation, analysis, report) is driven by one command,
`tools/run_pipeline.py`, reading the single settings file
`configs/run_profile.yaml`. The full field list and a walkthrough of running
the same pipeline locally (no notebook) are in `RUNBOOK.md`.

Resume state is *only* the output directory named in the profile: re-running
this notebook after an interruption is safe, and a model with every row
already written is skipped without even loading it.


### 1. Bootstrap: clone, authenticate, install

One-time per Colab runtime. Set `BRANCH` below only if you're testing a
feature branch instead of `main`.


In [ ]:
BRANCH = "main"  # change only to test a feature branch

!git clone -b {BRANCH} https://github.com/natSegOS/glamor-research-onboarding.git
%cd glamor-research-onboarding/experiments/001_typo_robustness

import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF token loaded from Colab secret HF_TOKEN")
except Exception:
    # Not on Colab, or no HF_TOKEN secret configured: fall back to an
    # interactive login. Needed for gated repos (e.g. llama_1b).
    from huggingface_hub import notebook_login
    notebook_login()

!python tools/run_pipeline.py setup


### 2. Configure: Colab-specific overrides

The committed `configs/run_profile.yaml` is the source of truth for the run
(experiment config, output namespace, model roster). This cell loads it and
overrides only what's genuinely Colab-specific — the Drive-backed output path
and the T4-safe model subset — so it can't drift out of sync with the file.

`python tools/run_pipeline.py list-models` (in a scratch cell) prints every
model key. To parallelize across Google accounts, give each session a `models`
override containing just the one model that account should run.


In [ ]:
import yaml
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/glamor"
except ImportError:
    DRIVE_ROOT = "."  # not on Colab: keep everything local to the clone

# The committed configs/run_profile.yaml is the single source of truth for the
# science: experiment config, output namespace (e.g. rehearsal_v3), data-reuse
# and skip policy. Load it and override ONLY the two things that are genuinely
# Colab-specific, so this cell can never silently fall behind the file again.
profile_path = Path("configs/run_profile.yaml")
profile = yaml.safe_load(profile_path.read_text())

# 1. Drive-back the output so results survive a runtime restart. Derived from
#    the committed output_root, so a v3 -> v4 bump in the file needs no edit here.
profile["output_root"] = f"{DRIVE_ROOT}/{profile['output_root']}"

# 2. Free-tier Colab is a 16 GB T4: it runs the small models and the 4-bit AWQ
#    7-8B builds, but NOT the full fp16 7-8B models (qwen_7b, llama_8b,
#    mistral_7b) that the committed 8-model roster also includes. Those three
#    run on the GPU cluster. Delete this override on an A100/high-RAM runtime
#    to run the full committed roster; to split the roster across Google
#    accounts, give each session a one-model list here.
profile["models"] = [
    "qwen_1b5_pilot", "llama_1b", "llama_3b", "qwen_7b_awq", "llama_8b_awq",
]

profile_path.write_text(yaml.safe_dump(profile, sort_keys=False))
print(yaml.safe_dump(profile, sort_keys=False))


### 3. Run

Prints a plan (what will run, what's already complete, whether data is being
reused) before touching any GPU, then runs data prep → generation → analysis
→ report. A model that fails (OOM, gated access) is recorded and skipped;
the rest of the roster continues.


In [ ]:
!python tools/run_pipeline.py all


### 4. Download results


In [ ]:
import zipfile
from pathlib import Path

output_root = Path(profile["output_root"])
archive_path = Path("rehearsal_results.zip")

with zipfile.ZipFile(archive_path, "w") as archive:
    for path in Path(profile["analysis_dir"]).rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to("."))
    for path in output_root.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(output_root.parent))

try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print(f"Saved to {archive_path.resolve()}")
